In [8]:
from pathlib import Path
import pandas as pd
from pandas import DataFrame

from helpers import rename_patient, boldify_index_and_columns

In [9]:
in_path = Path('~/thesis_files/statistical_results/.model_feature_qualifications.pkl').expanduser()
out_root = Path('~/Developer/MastersThesis/src/tables').expanduser()
cnn_out_path = out_root / 'cnn_feature_qualifications.tex'
ens_out_path = out_root / 'ensemble_feature_qualifications.tex'

In [10]:
orig = pd.read_pickle(in_path)
orig.head()

seizures                   CNN              \
metric                 p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                    
competition-1 corrcoef      0.415003       False  0.610688       False   
              acfw_D        0.000593        True  0.610688       False   
              acfw_P        0.019769        True  0.610688       False   
              var_D         0.880175       False  0.610688       False   
              var_P         0.035469        True  0.610688       False   

                                                                             \
metric                 p_rayleigh_bh significant p_perm_bh    met qualified   
patient       feature                                                         
competition-1 corrcoef  1.757952e-86        True  0.507299   True     False   
              acfw_D    9.179538e-39        True  0.001200  False     False   
              acfw_P    1.619894e-03        True  0.001200  False     False   
              var_D     4.197613e-07        True  0.402777   True     False   
              var_P     1.947527e-57        True  0.001666  False     False   

                       ensemble                                         \
metric                  roc_auc roc_auc_met  p_rayleigh_bh significant   
patient       feature                                                    
competition-1 corrcoef  0.59268       False  1.138508e-106        True   
              acfw_D    0.59268       False   1.440980e-56        True   
              acfw_P    0.59268       False   1.460106e-05        True   
              var_D     0.59268       False   2.902279e-06        True   
              var_P     0.59268       False   3.743980e-77        True   

                                                   
metric                 p_perm_bh    met qualified  
patient       feature                              
competition-1 corrcoef  0.526295   True     False  
              acfw_D    0.001333  False     False  
              acfw_P    0.001333  False     False  
              var_D     0.412418   True     False  
              var_P     0.001333  False     False

In [11]:
cnn = orig.drop(columns=['ensemble']).copy()
ens = orig.drop(columns=['CNN']).copy()
# cnn

In [12]:
cnn_qual = cnn[cnn[('CNN', 'qualified')]]
cnn_qual

seizures                   CNN              \
metric                 p_rayleigh_bh significant   roc_auc roc_auc_met   
patient       feature                                                    
competition-2 corrcoef  1.797378e-02        True  0.796782        True   
              acfw_D    1.574659e-02        True  0.796782        True   
U002-DE01-01  acfw_P    1.039236e-39        True  0.817384        True   

                                                                             
metric                  p_rayleigh_bh significant p_perm_bh   met qualified  
patient       feature                                                        
competition-2 corrcoef   1.159836e-23        True  0.085054  True      True  
              acfw_D    7.262569e-111        True  0.163167  True      True  
U002-DE01-01  acfw_P     0.000000e+00        True  0.127774  True      True

In [13]:
ens_qual = ens[ens[('ensemble', 'qualified')]]
ens_qual

seizures              ensemble              \
metric               p_rayleigh_bh significant   roc_auc roc_auc_met   
patient      feature                                                   
U002-DE01-01 acfw_P   1.039236e-39        True  0.789480        True   
U002-DE01-17 acfw_P   4.206776e-02        True  0.787532        True   
             var_D    2.869367e-06        True  0.787532        True   
             Delta_D  0.000000e+00        True  0.787532        True   
             Delta_P  0.000000e+00        True  0.787532        True   
             Alpha_P  2.605462e-03        True  0.787532        True   

                                                                          
metric               p_rayleigh_bh significant p_perm_bh   met qualified  
patient      feature                                                      
U002-DE01-01 acfw_P   0.000000e+00        True  0.959408  True      True  
U002-DE01-17 acfw_P   1.633171e-97        True  0.798994  True      True  
             var_D    1.938298e-13        True  0.871426  True      True  
             Delta_D  6.585703e-11        True  0.059088  True      True  
             Delta_P  6.119294e-08        True  0.118226  True      True  
             Alpha_P  2.165949e-13        True  0.118226  True      True

In [17]:
def apply_transformations(df: DataFrame, model: str) -> DataFrame:
    t = df.copy()

    # Abbreviate patients
    idx_df = t.index.to_frame(index=False)
    idx_df['patient'] = idx_df['patient'].map(rename_patient)
    t.index = pd.MultiIndex.from_frame(idx_df)

    t = t.drop(columns=[('seizures', 'significant')] + [(model, c) for c in
                                                        ['significant', 'roc_auc_met', 'met', 'qualified']])

    t = t.rename_axis(index={'patient': 'Patient', 'feature': 'Feature'},
                      columns={'metric': 'Metric'})
    t = t.rename(
        columns={
            'seizures': 'Seizures',
            'ensemble': 'Ensemble',

            'p_rayleigh_bh': r'$p_{\mathrm{Rayleigh, BH}}$',
            'significant': '$<0.05$',

            'roc_auc': 'ROC AUC',
            'roc_auc_met': r'$\geq 0.75$',

            'p_perm_bh': r'$p_{\mathrm{Permutation, BH}}$',
            'met': r'$\geq 0.05$',
        },
        index={
            'corrcoef': 'Corr. Coef.',
            'acfw_D': 'ACFW (D)',
            'acfw_P': 'ACFW (P)',
            'var_D': 'Var. (D)',
            'var_P': 'Var. (P)',
            **{f'{band}_P': f'{band} (P)' for band in ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']},
            **{f'{band}_D': f'{band} (D)' for band in ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']},
        }
    )
    # t = boldify_index_and_columns(t)
    return t


cnn_t = apply_transformations(cnn_qual, 'CNN')
ens_t = apply_transformations(ens_qual, 'ensemble')

cnn_t

Seizures       CNN  \
Metric              $p_{\mathrm{Rayleigh, BH}}$   ROC AUC   
Patient Feature                                             
C02     Corr. Coef.                1.797378e-02  0.796782   
        ACFW (D)                   1.574659e-02  0.796782   
U01     ACFW (P)                   1.039236e-39  0.817384   

                                                                                
Metric              $p_{\mathrm{Rayleigh, BH}}$ $p_{\mathrm{Permutation, BH}}$  
Patient Feature                                                                 
C02     Corr. Coef.                1.159836e-23                       0.085054  
        ACFW (D)                  7.262569e-111                       0.163167  
U01     ACFW (P)                   0.000000e+00                       0.127774

In [18]:
ens_t

Seizures  Ensemble  \
Metric            $p_{\mathrm{Rayleigh, BH}}$   ROC AUC   
Patient Feature                                           
U01     ACFW (P)                 1.039236e-39  0.789480   
U17     ACFW (P)                 4.206776e-02  0.787532   
        Var. (D)                 2.869367e-06  0.787532   
        Delta (D)                0.000000e+00  0.787532   
        Delta (P)                0.000000e+00  0.787532   
        Alpha (P)                2.605462e-03  0.787532   

                                                                              
Metric            $p_{\mathrm{Rayleigh, BH}}$ $p_{\mathrm{Permutation, BH}}$  
Patient Feature                                                               
U01     ACFW (P)                 0.000000e+00                       0.959408  
U17     ACFW (P)                 1.633171e-97                       0.798994  
        Var. (D)                 1.938298e-13                       0.871426  
        Delta (D)                6.585703e-11                       0.059088  
        Delta (P)                6.119294e-08                       0.118226  
        Alpha (P)                2.165949e-13                       0.118226

In [22]:
print(cnn_t.to_latex(float_format='%.3f', escape=False, multicolumn=True, multicolumn_format='|c', column_format='llr|rrr'))

\begin{tabular}{llr|rrr}
\toprule
 &  & Seizures & \multicolumn{3}{|c}{CNN} \\
 & Metric & $p_{\mathrm{Rayleigh, BH}}$ & ROC AUC & $p_{\mathrm{Rayleigh, BH}}$ & $p_{\mathrm{Permutation, BH}}$ \\
Patient & Feature &  &  &  &  \\
\midrule
\multirow[t]{2}{*}{C02} & Corr. Coef. & 0.018 & 0.797 & 0.000 & 0.085 \\
 & ACFW (D) & 0.016 & 0.797 & 0.000 & 0.163 \\
\cline{1-6}
U01 & ACFW (P) & 0.000 & 0.817 & 0.000 & 0.128 \\
\cline{1-6}
\bottomrule
\end{tabular}



In [23]:
print(ens_t.to_latex(float_format='%.3f', escape=False, multicolumn=True, multicolumn_format='|c', column_format='llr|rrr'))

\begin{tabular}{llr|rrr}
\toprule
 &  & Seizures & \multicolumn{3}{|c}{Ensemble} \\
 & Metric & $p_{\mathrm{Rayleigh, BH}}$ & ROC AUC & $p_{\mathrm{Rayleigh, BH}}$ & $p_{\mathrm{Permutation, BH}}$ \\
Patient & Feature &  &  &  &  \\
\midrule
U01 & ACFW (P) & 0.000 & 0.789 & 0.000 & 0.959 \\
\cline{1-6}
\multirow[t]{5}{*}{U17} & ACFW (P) & 0.042 & 0.788 & 0.000 & 0.799 \\
 & Var. (D) & 0.000 & 0.788 & 0.000 & 0.871 \\
 & Delta (D) & 0.000 & 0.788 & 0.000 & 0.059 \\
 & Delta (P) & 0.000 & 0.788 & 0.000 & 0.118 \\
 & Alpha (P) & 0.003 & 0.788 & 0.000 & 0.118 \\
\cline{1-6}
\bottomrule
\end{tabular}

